In [36]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns ,warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle 

In [37]:
## Load the dataset
df = pd.read_csv('Churn_Modelling.csv')
## dataset display
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [38]:
# remove3 unnecessary columns
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)
## dataset display
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [39]:
encode_gender = LabelEncoder()
df['Gender'] = encode_gender.fit_transform(df["Gender"])

In [40]:
from sklearn.preprocessing import OneHotEncoder
encode_geography = OneHotEncoder()
geo_encode = encode_geography.fit_transform(df[["Geography"]])
geo_encode

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [41]:
encode_geography.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [42]:
final = pd.DataFrame(geo_encode.toarray(),columns=encode_geography.get_feature_names_out(["Geography"]))
final.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [43]:
df = pd.concat([df.drop("Geography",axis=1),final],axis=1)

In [44]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [45]:
## save encode 
with open("gender_encode.pkl","wb") as  file:
    pickle.dump(encode_gender,file)

with open("geography_encode.pkl","wb") as file:
    pickle.dump(encode_geography,file)

In [46]:
X = df.drop("Exited",axis=1)
y = df[['Exited']]

In [47]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [48]:
## scaling data
scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [49]:
## save scale
with open("scale.pkl","wb") as file:
    pickle.dump(scale,file)

# ANN Implementation

In [50]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [51]:
## build ann model
model = Sequential(
    [
        Dense(64,activation="relu",input_shape=(X_train.shape[1],)), ## Hidden layer 1 connected to input layer
        Dense(32,activation="relu"), ## Hidden layer 2
        Dense(1,activation="sigmoid") ## output layer
    ]
)

In [52]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [53]:
## initialize optimizer
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
binary_loss = tensorflow.keras.losses.BinaryCrossentropy()

In [54]:
## compile the model
# model.compile(optimizer="adam", loss="binary_crossentropy",metrics=['accuracy'])
model.compile(optimizer=opt, loss= binary_loss ,metrics=['accuracy'])

In [55]:
## setup the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [56]:
## setup early stop
early_stopin_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [57]:
## train model
history = model.fit(
    X_train,y_train,validation_data = (X_test,y_test), epochs=100,
    callbacks = [tensorflow_callback,early_stopin_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8339 - loss: 0.3972 - val_accuracy: 0.8465 - val_loss: 0.3664
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8570 - loss: 0.3522 - val_accuracy: 0.8590 - val_loss: 0.3423
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8616 - loss: 0.3458 - val_accuracy: 0.8510 - val_loss: 0.3600
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8584 - loss: 0.3438 - val_accuracy: 0.8560 - val_loss: 0.3518
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8584 - loss: 0.3402 - val_accuracy: 0.8560 - val_loss: 0.3438
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8626 - loss: 0.3366 - val_accuracy: 0.8600 - val_loss: 0.3427
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8620 - loss: 0.3335 - val_accuracy: 0.8600 - val_loss: 0.3412
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8645 - loss: 0.3322 - val_accu

In [58]:
model.save('model.h5')

In [61]:
## load tensorbord extension
%load_ext tensorboard 

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [62]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6007 (pid 2772), started 3:19:32 ago. (Use '!kill 2772' to kill it.)

_____